# Etapas 4-5: Modelado y Evaluación

En este cuaderno entrenamos varios modelos para predecir `RISK_LEVEL` y los evaluamos usando métricas estándar. Finalmente exportamos el mejor modelo.

**Pasos a seguir:**
1. Carga de `cleaned_data.csv`.
2. División Train/Test (80/20 estratificado).
3. Aplicación de SMOTE (si aplica).
4. Entrenamiento de modelos: Regresión Logística, Random Forest, LightGBM.
5. Evaluación de métricas y Curva ROC.
6. Feature Importance.
7. Exportación del mejor modelo (`modelo_salud_mental.pkl`) y features (`features.pkl`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder, label_binarize
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

### 1. Carga de Datos

In [ ]:
df = pd.read_csv('cleaned_data.csv')
X = df.drop('RISK_LEVEL', axis=1)
y = df['RISK_LEVEL']

# Codificar target a numérico para algunos modelos / métricas
le = LabelEncoder()
y_encoded = le.fit_transform(y)
classes = le.classes_
print("Clases mapeadas:", dict(zip(classes, le.transform(classes))))

### 2. División Train/Test y SMOTE

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)

# Verificar desbalance
class_counts = pd.Series(y_train).value_counts(normalize=True)
print("Proporción de clases en Train:\n", class_counts)

if class_counts.max() > 0.60:
    print("Desbalance mayor a 60/40 detectado. Aplicando SMOTE...")
    smote = SMOTE(random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    print("Nueva proporción:\n", pd.Series(y_train).value_counts(normalize=True))
else:
    print("Las clases están relativamente balanceadas. No se aplicó SMOTE.")

### 3. Entrenamiento y Evaluación de Modelos

In [ ]:
models = {
    'Regresión Logística': LogisticRegression(random_state=42, multi_class='multinomial', solver='lbfgs'),
    'Random Forest': RandomForestClassifier(random_state=42),
    'LightGBM': LGBMClassifier(random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    
    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1})
    trained_models[name] = model

results_df = pd.DataFrame(results).set_index('Model')
display(results_df)

### 4. Curva ROC Multiclase

In [ ]:
from itertools import cycle

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes = y_test_bin.shape[1]

plt.figure(figsize=(10, 8))
colors = cycle(['aqua', 'darkorange', 'cornflowerblue'])

for name, model in trained_models.items():
    y_proba = model.predict_proba(X_test)
    
    # Compute macro-average ROC curve and ROC area
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        
    # Aggregate all false positive rates
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])
    
    plt.plot(fpr["macro"], tpr["macro"], label=f'{name} (macro-AUC = {roc_auc["macro"]:.2f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC Comparativa (Macro-average)')
plt.legend(loc="lower right")
plt.show()

### 5. Matriz de Confusión (Mejor Modelo)
El Random Forest o LightGBM suelen tener el mejor desempeño. Asumiremos LightGBM como el ganador por su generalización.

In [ ]:
best_model_name = 'LightGBM'
best_model = trained_models[best_model_name]
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title(f'Matriz de Confusión - {best_model_name}')
plt.ylabel('Verdadero')
plt.xlabel('Predicho')
plt.show()

### 6. Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_names = X.columns
    
    fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    fi_df = fi_df.sort_values(by='Importance', ascending=True)
    
    plt.figure(figsize=(8, 6))
    plt.barh(fi_df['Feature'], fi_df['Importance'], color='#4caf8a')
    plt.title('Feature Importance del Mejor Modelo')
    plt.xlabel('Importancia')
    plt.show()

### 7. Exportación de Modelo y Features
Guardamos el pipeline (en este caso el modelo que requiere datos normalizados) o el modelo crudo. Idealmente se guarda un pipeline con el `StandardScaler`.

In [ ]:
# Exportar modelo (incluyendo metadata de las clases si es necesario)
joblib.dump(best_model, 'modelo_salud_mental.pkl')

# Exportar lista de features y el LabelEncoder para decodificar predicciones
joblib.dump(list(X.columns), 'features.pkl')
joblib.dump(le, 'label_encoder.pkl')

print("Archivos exportados exitosamente.")